# 🌱 Agri-AI: Pest Classification & Detection Training (Colab GPU Edition)

This notebook trains a **YOLOv8 Nano Classifier (`yolov8n-cls`)** on the Agri-AI Pest Dataset in ~3 minutes using free Google Colab GPU (T4).

### Pipeline Steps:
1. Install Ultralytics
2. Upload & unzip dataset
3. Train YOLOv8 with real-time augmentations (flips, mosaic, color jitter)
4. Evaluate top-1 & top-5 accuracy on Test split
5. Export to ONNX (`pest_detector.onnx`)
6. Download the ONNX file for your Agri-AI backend

In [ ]:
# Step 1: Install Ultralytics
!pip install -q ultralytics onnx

In [ ]:
# Step 2: Verify GPU
import torch
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Step 3: Upload your cleaned split dataset or download from Google Drive
# You can zip your 'pest-dataset/split' folder and upload it here:
from google.colab import files
import zipfile, os

if not os.path.exists('split'):
    print('Please upload split.zip...')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as z:
                z.extractall('.')
            print('Extracted successfully!')

In [ ]:
# Step 4: Train YOLOv8 Nano Classifier
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')

results = model.train(
    data='split',
    epochs=25,
    imgsz=224,
    batch=16,
    project='runs',
    name='pest_model'
)

In [ ]:
# Step 5: Evaluate on Test Split
metrics = model.val(data='split', split='test')
print(f'Test Top-1 Accuracy: {metrics.top1:.4f}')
print(f'Test Top-5 Accuracy: {metrics.top5:.4f}')

In [ ]:
# Step 6: Export Best Model to ONNX
onnx_path = model.export(format='onnx', imgsz=224)
print('ONNX model ready at:', onnx_path)

# Rename and download directly to computer
import shutil
shutil.copyfile(onnx_path, 'pest_detector.onnx')
files.download('pest_detector.onnx')
print('Download started! Drop pest_detector.onnx into backend/pest-model/')